In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Alipur_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,320.0,173.0,200.0,105.0,189.0,292.0,94.0,40.0,80.0,127.0,339.0,286.0
1,2,308.0,167.0,100.0,129.0,169.0,225.0,109.0,63.0,63.0,175.0,326.0,272.0
2,3,294.0,174.0,NaN,166.0,245.0,233.0,102.0,54.0,NaN,161.0,384.0,278.0
3,4,357.0,221.0,99.0,114.0,256.0,195.0,49.0,47.0,43.0,NaN,400.0,159.0
4,5,281.0,177.0,97.0,149.0,315.0,274.0,56.0,25.0,NaN,150.0,380.0,151.0
5,6,302.0,115.0,93.0,168.0,225.0,249.0,37.0,52.0,88.0,135.0,376.0,210.0
6,7,342.0,118.0,153.0,156.0,318.0,NaN,36.0,50.0,37.0,119.0,392.0,244.0
7,8,325.0,129.0,120.0,194.0,227.0,223.0,41.0,39.0,66.0,NaN,390.0,314.0
8,9,267.0,113.0,98.0,192.0,191.0,271.0,58.0,35.0,96.0,162.0,380.0,162.0
9,10,227.0,261.0,129.0,274.0,221.0,258.0,154.0,47.0,106.0,114.0,349.0,236.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   34 non-null     float64
 3   March      34 non-null     float64
 4   April      34 non-null     float64
 5   May        34 non-null     float64
 6   June       34 non-null     float64
 7   July       35 non-null     float64
 8   August     34 non-null     float64
 9   September  34 non-null     float64
 10  October    34 non-null     float64
 11  November   33 non-null     float64
 12  December   36 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,320.000000,173.000000,200.000000,105.000000,189.000000,292.000000,94.000000,40.000000,80.000000,127.000000,339.000000,286.000000
1,2,308.000000,167.000000,100.000000,129.000000,169.000000,225.000000,109.000000,63.000000,63.000000,175.000000,326.000000,272.000000
2,3,294.000000,174.000000,136.852941,166.000000,245.000000,233.000000,102.000000,54.000000,81.264706,161.000000,384.000000,278.000000
3,4,357.000000,221.000000,99.000000,114.000000,256.000000,195.000000,49.000000,47.000000,43.000000,213.529412,400.000000,159.000000
4,5,281.000000,177.000000,97.000000,149.000000,315.000000,274.000000,56.000000,25.000000,81.264706,150.000000,380.000000,151.000000
5,6,302.000000,115.000000,93.000000,168.000000,225.000000,249.000000,37.000000,52.000000,88.000000,135.000000,376.000000,210.000000
6,7,342.000000,118.000000,153.000000,156.000000,318.000000,169.352941,36.000000,50.000000,37.000000,119.000000,392.000000,244.000000
7,8,325.000000,129.000000,120.000000,194.000000,227.000000,223.000000,41.000000,39.000000,66.000000,213.529412,390.000000,314.000000
8,9,267.000000,113.000000,98.000000,192.000000,191.000000,271.000000,58.000000,35.000000,96.000000,162.000000,380.000000,162.000000
9,10,227.000000,261.000000,129.000000,274.000000,221.000000,258.000000,154.000000,47.000000,106.000000,114.000000,349.000000,236.000000
